In [141]:
from pathlib import Path
import hashlib
import json
import math
import os
import re
import sqlite3
import subprocess
from collections import Counter


# ============================================================
# PROJECT PATHS
# ============================================================

ROOT = Path(r"C:\amrita_uni\s6\NLP\project\Rubric-based-evaluation-of-PL-SQL-code\Rubric-based-evaluation-of-PL-SQL-code")

CACHE_DIR = ROOT / ".cache" / "notebook_ollama"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = ROOT / "artifacts" / "schema_output"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "qwen2.5:7b"

PROBLEM_PATH = ROOT / "examples" / "bank_problem.txt"


# ============================================================
# LOAD PROBLEM STATEMENT
# ============================================================

problem_statement = (
    PROBLEM_PATH.read_text(encoding="utf-8")
    if PROBLEM_PATH.exists()
    else
    '''
    Consider a Bank database which includes the following tables.

    ACCOUNTS(ac_no, br_no, cust_no, ac_type, bal)
    BRANCHES(br_no, br_name, loc)
    CUSTOMER(cno, cname, c_type)

    1. Write a function that accepts a threshold value and a customer number.
       The program updates the c_type based on the threshold value.
       If balance > threshold then class A, else class B.

    2. Write a function called CloseBranch that takes two arguments
       (the branch to be closed and the branch to take over the accounts)
       and transfers all accounts at the closing branch to the new branch
       and removes the closing branch.

    3. Write a function that implements a safe withdrawal operation,
       that only permits a withdraw if there are sufficient funds
       in the account to cover it.
    '''
)

print(problem_statement)

Consider a Bank database which includes the following tables.

ACCOUNTS(ac_no, br_no, cust_no, ac_type, bal)
BRANCHES(br_no, br_name, loc)
CUSTOMER(cno, cname, c_type)

1. Write a function that accepts a threshold value and a customer number. The program updates the c_type based on the threshold value. If balance > threshold then class A, else class B.
2. Write a function called CloseBranch that takes two arguments (the branch to be closed and the branch to take over the accounts) and transfers all accounts at the closing branch to the new branch and removes the closing branch.
3. Write a function that implements a safe withdrawal operation, that only permits a withdraw if there are sufficient funds in the account to cover it.



In [142]:
def normalize_identifier(value: str) -> str:
    """
    Convert identifiers into normalized snake_case style.
    """
    return re.sub(r"[^a-z0-9_]+", "_", value.strip().lower()).strip("_")


def tokenize(text: str) -> list[str]:
    """
    Lightweight tokenizer.
    """
    return re.findall(r"[a-zA-Z_][a-zA-Z0-9_]+", text.lower())


def count_terms(tokens: list[str]) -> dict:
    """
    Create term-frequency dictionary.
    """
    counts = {}

    for token in tokens:
        counts[token] = counts.get(token, 0) + 1

    return counts


def cosine_similarity(left: dict, right: dict) -> float:
    """
    Compute cosine similarity between sparse vectors.
    """

    if not left or not right:
        return 0.0

    numerator = sum(
        left.get(token, 0.0) * right.get(token, 0.0)
        for token in left
    )

    left_norm = math.sqrt(sum(v * v for v in left.values()))
    right_norm = math.sqrt(sum(v * v for v in right.values()))

    if not left_norm or not right_norm:
        return 0.0

    return numerator / (left_norm * right_norm)

In [143]:
def clean_cli_output(text: str) -> str:
    """
    Remove markdown wrappers and terminal artifacts.
    """

    text = re.sub(r"\x1b\[[0-9;]*m", "", text)

    text = text.replace("```json", "")
    text = text.replace("```", "")

    return text.strip()


def ollama_chat(messages: list[dict], model: str = MODEL) -> str:
    """
    Cached Ollama chat interface.
    """

    payload = json.dumps(messages, sort_keys=True)

    cache_key = hashlib.sha256(payload.encode()).hexdigest()

    cache_path = CACHE_DIR / f"{cache_key}.txt"

    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8")

    full_prompt = "\n\n".join(
        [
            f"{m['role'].upper()}:\n{m['content']}"
            for m in messages
        ]
    )

    result = subprocess.run(
        ["ollama", "run", model],
        input=full_prompt,
        text=True,
        capture_output=True,
        encoding="utf-8"
    )

    output = clean_cli_output(result.stdout)

    cache_path.write_text(output, encoding="utf-8")

    return output

In [144]:
# ============================================================
# SECTION 4 — GENERALIZED PROCEDURAL ANALYSIS
# ============================================================

GENERALIZED_PROCEDURAL_PATTERNS = {

    "transaction_sensitive_operation": [
        "withdraw",
        "deposit",
        "transfer",
        "payment",
        "purchase",
        "booking",
        "update",
        "rollback",
        "commit"
    ],

    "boundary_validation": [
        "threshold",
        "limit",
        "minimum",
        "maximum",
        "greater",
        "less",
        "equal",
        "range"
    ],

    "cross_table_dependency": [
        "transfer",
        "move",
        "delete",
        "dependency",
        "relationship",
        "foreign key",
        "reference"
    ],

    "classification_logic": [
        "classify",
        "categorize",
        "grade",
        "assign"
    ],

    "exception_sensitive_operation": [
        "exception",
        "error",
        "invalid",
        "failure",
        "raise"
    ],

    "null_sensitive_operation": [
        "null",
        "empty",
        "missing"
    ]
}


GENERALIZED_RISK_PATTERNS = {

    "rollback_failure": [
        "transaction",
        "rollback",
        "commit"
    ],

    "threshold_logic_failure": [
        "threshold",
        "comparison",
        "range"
    ],

    "dependency_violation": [
        "dependency",
        "foreign key",
        "relationship"
    ],

    "null_handling_failure": [
        "null",
        "empty",
        "missing"
    ],

    "exception_swallowing": [
        "exception",
        "raise"
    ]
}


def extract_generalized_intents(problem_text: str) -> dict:
    """
    Deterministic procedural semantic extraction layer.
    """

    lower = problem_text.lower()

    procedural_patterns = []

    for pattern_name, keywords in GENERALIZED_PROCEDURAL_PATTERNS.items():

        if any(keyword in lower for keyword in keywords):
            procedural_patterns.append(pattern_name)

    risk_patterns = []

    for risk_name, keywords in GENERALIZED_RISK_PATTERNS.items():

        if any(keyword in lower for keyword in keywords):
            risk_patterns.append(risk_name)

    behavioral_expectations = []

    if "withdraw" in lower:

        behavioral_expectations.extend([

            "Operation should reject insufficient balance.",

            "Exact balance withdrawals should succeed.",

            "Negative withdrawal amounts should fail."
        ])

    if "threshold" in lower:

        behavioral_expectations.extend([

            "Boundary equality cases must be tested.",

            "Threshold minus one cases should behave correctly.",

            "Threshold plus one cases should behave correctly."
        ])

    if "transfer" in lower or "branch" in lower:

        behavioral_expectations.extend([

            "Cross-table updates should preserve consistency.",

            "Partial failures should rollback safely."
        ])

    return {

        "procedural_patterns":
            sorted(set(procedural_patterns)),

        "risk_patterns":
            sorted(set(risk_patterns)),

        "behavioral_expectations":
            sorted(set(behavioral_expectations))
    }

In [145]:
def extract_entity_hints(problem_text: str) -> list[dict]:
    """
    Extract table/entity definitions from problem statement.
    """

    table_defs = re.findall(
        r"([A-Za-z_]+)\(([^\)]*)\)",
        problem_text
    )

    entity_hints = []

    for table_name, columns in table_defs:

        cols = [
            col.strip()
            for col in columns.split(",")
            if col.strip()
        ]

        entity_hints.append({
            "name": table_name.upper(),
            "columns": cols
        })

    return entity_hints

In [146]:
def decompose_problem(problem_text: str) -> dict:
    """
    Full generalized decomposition layer.
    """

    generalized_analysis = extract_generalized_intents(problem_text)

    return {

        "entity_hints":
            extract_entity_hints(problem_text),

        "procedural_patterns":
            generalized_analysis["procedural_patterns"],

        "risk_patterns":
            generalized_analysis["risk_patterns"],

        "behavioral_expectations":
            generalized_analysis["behavioral_expectations"],

        "prompt_token_count":
            len(tokenize(problem_text))
    }


prompt_decomposition = decompose_problem(problem_statement)

print(json.dumps(prompt_decomposition, indent=2))

{
  "entity_hints": [
    {
      "name": "ACCOUNTS",
      "columns": [
        "ac_no",
        "br_no",
        "cust_no",
        "ac_type",
        "bal"
      ]
    },
    {
      "name": "BRANCHES",
      "columns": [
        "br_no",
        "br_name",
        "loc"
      ]
    },
    {
      "name": "CUSTOMER",
      "columns": [
        "cno",
        "cname",
        "c_type"
      ]
    }
  ],
  "procedural_patterns": [
    "boundary_validation",
    "cross_table_dependency",
    "transaction_sensitive_operation"
  ],
  "risk_patterns": [
    "threshold_logic_failure"
  ],
  "behavioral_expectations": [
    "Boundary equality cases must be tested.",
    "Cross-table updates should preserve consistency.",
    "Exact balance withdrawals should succeed.",
    "Negative withdrawal amounts should fail.",
    "Operation should reject insufficient balance.",
    "Partial failures should rollback safely.",
    "Threshold minus one cases should behave correctly.",
    "Threshold plu

In [147]:
SCHEMA_SYSTEM_PROMPT = """
You are an expert Oracle PL/SQL schema designer.

Generate STRICT VALID JSON.

Requirements:

1. Use Oracle-friendly schema design.
2. Detect:
   - entities
   - relationships
   - constraints
   - procedural semantics

3. Capture:
   - transaction-sensitive operations
   - boundary-sensitive logic
   - cross-table dependencies
   - classification logic
   - exception-sensitive operations

4. Output ONLY VALID JSON.

Required output structure:

{
  "title": "...",

  "entities": [
    {
      "name": "...",

      "attributes": [
        {
          "name": "...",
          "type": "...",
          "nullable": false
        }
      ],

      "primary_key": ["..."]
    }
  ],

  "relationships": [
    {
      "from_entity": "...",
      "to_entity": "...",
      "type": "many_to_one"
    }
  ],

  "constraints": [],

  "procedural_intents": [
    {
      "name": "...",
      "risk_tags": [],
      "behavioral_expectations": []
    }
  ]
}
"""

In [148]:
def clean_json_blob(text: str) -> str:
    """
    Extract clean JSON object from model output.
    """

    text = clean_cli_output(text)

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON found.")

    return text[start:end+1]

In [149]:
# ============================================================
# SECTION 8A — INTENT NORMALIZATION + AUGMENTATION
# ============================================================

INTENT_NORMALIZATION_MAP = {

    "safe_withdrawal":
        "transaction_sensitive_operation",

    "withdrawal":
        "transaction_sensitive_operation",

    "branch_transfer":
        "cross_table_dependency",

    "transfer_operation":
        "cross_table_dependency",

    "threshold_classification":
        "boundary_validation",

    "threshold_logic":
        "boundary_validation",

    "exception_handling":
        "exception_sensitive_operation",

    "null_handling":
        "null_sensitive_operation"
}


def normalize_intent_name(intent_name: str) -> str:
    """
    Normalize raw LLM intent names into
    generalized procedural patterns.
    """

    normalized = normalize_identifier(intent_name)

    return INTENT_NORMALIZATION_MAP.get(
        normalized,
        normalized
    )


def extract_llm_intents(schema: dict) -> list[str]:
    """
    Extract procedural intents generated by LLM.
    """

    intents = []

    for intent in schema.get("procedural_intents", []):

        if isinstance(intent, dict):

            name = intent.get("name", "")

            if name:

                intents.append(
                    normalize_intent_name(name)
                )

    return sorted(set(intents))


def validate_intent_coverage(
    llm_intents: list[str],
    decomposition: dict
) -> dict:
    """
    Validate procedural semantic coverage.
    """

    deterministic_intents = set(
        decomposition["procedural_patterns"]
    )

    llm_intents = set(llm_intents)

    missing = deterministic_intents - llm_intents

    coverage = (
        len(deterministic_intents & llm_intents)
        /
        max(1, len(deterministic_intents))
    )

    return {

        "coverage_score":
            round(coverage, 3),

        "missing_intents":
            sorted(missing)
    }


def merge_procedural_intents(
    schema: dict,
    decomposition: dict
) -> list[dict]:
    """
    Merge:
    - LLM semantic discovery
    - deterministic semantic guarantees
    """

    llm_intents = extract_llm_intents(schema)

    coverage_report = validate_intent_coverage(
        llm_intents,
        decomposition
    )

    final_intents = set(llm_intents)

    # ========================================================
    # AUGMENT MISSING SEMANTICS
    # ========================================================

    for missing_intent in coverage_report["missing_intents"]:

        final_intents.add(missing_intent)

    merged = []

    for intent_name in sorted(final_intents):

        merged.append({

            "name": intent_name,

            "risk_tags":
                decomposition["risk_patterns"],

            "behavioral_expectations":
                decomposition["behavioral_expectations"]
        })

    return merged

In [150]:
def heuristic_fallback_schema(
    decomposition: dict
) -> dict:
    """
    Deterministic fallback schema if LLM fails.
    """

    entities = []

    for entity_hint in decomposition["entity_hints"]:

        attrs = []

        for col in entity_hint["columns"]:

            attrs.append({
                "name": col,
                "type": "VARCHAR2(100)",
                "nullable": False
            })

        entities.append({
            "name": entity_hint["name"],
            "attributes": attrs,
            "primary_key": [entity_hint["columns"][0]]
        })

    procedural_intents = []

    for pattern in decomposition["procedural_patterns"]:

        procedural_intents.append({
            "name": pattern,
            "risk_tags": decomposition["risk_patterns"],
            "behavioral_expectations":
                decomposition["behavioral_expectations"]
        })

    return {
        "title": "Fallback Generated Schema",
        "entities": entities,
        "relationships": [],
        "constraints": [],
        "procedural_intents": procedural_intents
    }

In [151]:
# ============================================================
# SECTION 10 — SCHEMA EXTRACTION
# ============================================================

def extract_schema(
    problem_text: str,
    decomposition: dict
) -> dict:
    """
    Main schema extraction orchestration layer.
    """

    messages = [

        {
            "role": "system",
            "content": SCHEMA_SYSTEM_PROMPT
        },

        {
            "role": "user",
            "content":
                f"""
                Problem Statement:
                {problem_text}

                Decomposition:
                {json.dumps(decomposition, indent=2)}

                IMPORTANT:
                Output STRICT VALID JSON ONLY.
                No markdown.
                No explanations.
                """
        }
    ]

    try:

        print("Calling Ollama...")

        response = ollama_chat(messages)

        print("Response received.")

        cleaned = clean_json_blob(response)

        schema = json.loads(cleaned)

        # ====================================================
        # MERGE LLM + DETERMINISTIC PROCEDURAL INTENTS
        # ====================================================

        schema["procedural_intents"] = (
            merge_procedural_intents(
                schema,
                decomposition
            )
        )

        return schema

    except Exception as exc:

        print("LLM schema extraction failed.")
        print(exc)

        return heuristic_fallback_schema(decomposition)

In [152]:
def validate_schema(
    schema: dict,
    problem_text: str,
    decomposition: dict
) -> list[dict]:
    """
    Structural validation layer.
    """

    issues = []

    entities = schema.get("entities", [])

    if not entities:

        issues.append({
            "severity": "error",
            "stage": "entities",
            "code": "NO_ENTITIES",
            "message": "Schema contains no entities."
        })

    entity_names = {
        e["name"].upper()
        for e in entities
    }

    for entity in entities:

        if not entity.get("attributes"):

            issues.append({
                "severity": "error",
                "stage": "attributes",
                "code": "EMPTY_ENTITY",
                "message":
                    f"{entity['name']} has no attributes."
            })

        pk = entity.get("primary_key", [])

        if not pk:

            issues.append({
                "severity": "warning",
                "stage": "primary_key",
                "code": "MISSING_PK",
                "message":
                    f"{entity['name']} missing primary key."
            })

    return issues

In [153]:
def generate_oracle_ddl(schema: dict) -> str:
    """
    Generate Oracle-compatible DDL.
    """

    ddl = []

    for entity in schema.get("entities", []):

        lines = []

        for attr in entity["attributes"]:

            line = f"{attr['name']} {attr['type']}"

            if not attr.get("nullable", True):
                line += " NOT NULL"

            lines.append(line)

        pk = entity.get("primary_key", [])

        if pk:
            lines.append(
                f"PRIMARY KEY ({', '.join(pk)})"
            )

        create_stmt = f"""
CREATE TABLE {entity['name']} (
    {', '.join(lines)}
);
"""

        ddl.append(create_stmt)

    return "\n".join(ddl)

In [154]:
def oracle_to_sqlite(ddl: str) -> str:
    """
    Convert Oracle-friendly DDL into SQLite-compatible DDL.
    """

    ddl = ddl.replace("VARCHAR2", "TEXT")
    ddl = ddl.replace("NUMBER", "REAL")

    return ddl


def ddl_is_feasible(ddl_text: str):

    try:

        sqlite_ddl = oracle_to_sqlite(ddl_text)

        conn = sqlite3.connect(":memory:")

        conn.executescript(sqlite_ddl)

        conn.close()

        return True, "DDL executed successfully."

    except Exception as exc:

        return False, str(exc)

In [155]:
def schema_confidence(
    schema: dict,
    validation_issues: list[dict],
    ddl_ok: bool
) -> dict:
    """
    Estimate schema confidence.
    """

    score = 1.0

    score -= 0.1 * sum(
        1
        for issue in validation_issues
        if issue["severity"] == "warning"
    )

    score -= 0.2 * sum(
        1
        for issue in validation_issues
        if issue["severity"] == "error"
    )

    if not ddl_ok:
        score -= 0.3

    score = max(0.0, min(1.0, score))

    return {
        "confidence_score": round(score, 3),
        "ddl_feasible": ddl_ok,
        "issue_count": len(validation_issues)
    }

In [156]:
schema = extract_schema(
    problem_statement,
    prompt_decomposition
)

validation_issues = validate_schema(
    schema,
    problem_statement,
    prompt_decomposition
)

ddl_text = generate_oracle_ddl(schema)

ddl_ok, ddl_message = ddl_is_feasible(ddl_text)

confidence = schema_confidence(
    schema,
    validation_issues,
    ddl_ok
)

print(json.dumps(schema, indent=2))

print("\nDDL feasibility:", ddl_ok)
print("DDL message:", ddl_message)

print("\nConfidence:")
print(json.dumps(confidence, indent=2))

Calling Ollama...
Response received.
{
  "title": "Bank Database Schema",
  "entities": [
    {
      "name": "ACCOUNTS",
      "attributes": [
        {
          "name": "ac_no",
          "type": "NUMBER",
          "nullable": false
        },
        {
          "name": "br_no",
          "type": "NUMBER",
          "nullable": false
        },
        {
          "name": "cust_no",
          "type": "NUMBER",
          "nullable": false
        },
        {
          "name": "ac_type",
          "type": "VARCHAR2(10)",
          "nullable": false
        },
        {
          "name": "bal",
          "type": "NUMBER",
          "nullable": false
        }
      ],
      "primary_key": [
        "ac_no"
      ]
    },
    {
      "name": "BRANCHES",
      "attributes": [
        {
          "name": "br_no",
          "type": "NUMBER",
          "nullable": false
        },
        {
          "name": "br_name",
          "type": "VARCHAR2(50)",
          "nullable": false
       

In [157]:
# ============================================================
# LLM OPERATION EXTRACTION
# ============================================================

OPERATION_EXTRACTION_PROMPT = """
You are analyzing a PL/SQL programming assignment.

Identify every procedure or function students are expected to implement.

IMPORTANT:

Allowed semantic_categories:

- boundary_validation
- transaction_sensitive_operation
- cross_table_dependency
- exception_sensitive_operation
- null_sensitive_operation
- classification_logic

You MUST ONLY use values from the list above.

Return ONLY VALID JSON.

Format:

[
  {
    "operation_name":"...",
    "operation_type":"function",

    "semantic_categories":[
      ...
    ],

    "parameters":[
      {
        "name":"...",
        "type":"..."
      }
    ],

    "expected_effects":[
      ...
    ]
  }
]

Do not return explanations.
"""


def clean_json_array(text: str) -> str:

    text = clean_cli_output(text)

    start = text.find("[")
    end = text.rfind("]")

    if start == -1 or end == -1:
        raise ValueError(
            "No JSON array found."
        )

    return text[start:end+1]


def extract_operations_with_llm(
    problem_text: str,
    decomposition: dict
) -> list[dict]:

    messages = [

        {
            "role": "system",

            "content":
                OPERATION_EXTRACTION_PROMPT
        },

        {
            "role": "user",

            "content":
                f"""
Problem Statement:

{problem_text}

Detected procedural patterns:

{json.dumps(
    decomposition["procedural_patterns"],
    indent=2
)}

Use ONLY the above procedural patterns
for semantic_categories.
"""
        }
    ]

    try:

        response = ollama_chat(
            messages
        )

        cleaned = clean_json_array(
            response
        )

        operations = json.loads(
            cleaned
        )

        if isinstance(
            operations,
            list
        ):
            return operations

    except Exception as exc:

        print(
            "Operation extraction failed."
        )

        print(exc)

    return []

In [158]:
# ============================================================
# OPERATION EXTRACTION + VALIDATION
# ============================================================

VALID_SEMANTICS = {

    "boundary_validation",

    "transaction_sensitive_operation",

    "cross_table_dependency",

    "exception_sensitive_operation",

    "null_sensitive_operation",

    "classification_logic"
}

operations = (

    extract_operations_with_llm(

        problem_statement,

        prompt_decomposition
    )
)

# ============================================================
# CLEAN INVALID SEMANTICS
# ============================================================

for operation in operations:

    operation["semantic_categories"] = [

        category

        for category in operation.get(
            "semantic_categories",
            []
        )

        if category in VALID_SEMANTICS
    ]

print("\nOperations:\n")

print(
    json.dumps(
        operations,
        indent=2
    )
)

# ============================================================
# BUILD SEMANTIC → OPERATION MAP
# ============================================================

operation_behavior_map = {}

for operation in operations:

    operation_name = (
        operation["operation_name"]
    )

    semantic_categories = (
        operation.get(
            "semantic_categories",
            []
        )
    )

    for category in semantic_categories:

        operation_behavior_map[
            category
        ] = operation_name

    # ======================================
    # SPECIAL AUGMENTATION
    # ======================================

    if (
        "classification_logic"
        in
        semantic_categories
    ):

        operation_behavior_map[
            "boundary_validation"
        ] = operation_name

print(
    json.dumps(
        operation_behavior_map,
        indent=2
    )
)


Operations:

[
  {
    "operation_name": "UpdateCustomerClass",
    "operation_type": "function",
    "semantic_categories": [
      "classification_logic"
    ],
    "parameters": [
      {
        "name": "threshold_value",
        "type": "NUMBER"
      },
      {
        "name": "cust_no",
        "type": "NUMBER"
      }
    ],
    "expected_effects": [
      "Updates the c_type of a customer based on their account balance relative to a threshold value."
    ]
  },
  {
    "operation_name": "CloseBranch",
    "operation_type": "function",
    "semantic_categories": [
      "transaction_sensitive_operation",
      "cross_table_dependency"
    ],
    "parameters": [
      {
        "name": "closing_branch_no",
        "type": "NUMBER"
      },
      {
        "name": "new_branch_no",
        "type": "NUMBER"
      }
    ],
    "expected_effects": [
      "Transfers all accounts from the closing branch to a new branch, and removes the closing branch."
    ]
  },
  {
    "operation_n

In [159]:
(ARTIFACT_DIR / "schema.json").write_text(
    json.dumps(schema, indent=2),
    encoding="utf-8"
)

(ARTIFACT_DIR / "ddl.sql").write_text(
    ddl_text,
    encoding="utf-8"
)

(ARTIFACT_DIR / "prompt_decomposition.json").write_text(
    json.dumps(prompt_decomposition, indent=2),
    encoding="utf-8"
)

(ARTIFACT_DIR / "schema_confidence.json").write_text(
    json.dumps(confidence, indent=2),
    encoding="utf-8"
)

(
    ARTIFACT_DIR /
    "operations.json"
).write_text(

    json.dumps(
        operations,
        indent=2
    ),

    encoding="utf-8"
)
(
    ARTIFACT_DIR /
    "operation_behavior_map.json"
).write_text(

    json.dumps(
        operation_behavior_map,
        indent=2
    ),

    encoding="utf-8"
)
print("Artifacts exported.")



Artifacts exported.


In [160]:
# ============================================================
# SECTION 17 — ASSERTION TESTS
# ============================================================

entity_names = {
    entity["name"].upper()
    for entity in schema.get("entities", [])
}

intent_names = {
    intent["name"]
    for intent in schema.get("procedural_intents", [])
}

assert "ACCOUNTS" in entity_names
assert "CUSTOMER" in entity_names

assert (
    "transaction_sensitive_operation"
    in intent_names
)

assert (
    "boundary_validation"
    in intent_names
)

assert ddl_ok

assert confidence["confidence_score"] >= 0.65

print("Schema notebook tests passed.")

Schema notebook tests passed.
